In [ ]:
fases_workload = [
    '1. Carga Inicial\n(SELECT supplier)', 
    '2. Releitura\n(SELECT supplier)', 
    '3. Sequential Flooding\n(SELECT lineitem)', 
    '4. Prova de Retenção\n(SELECT supplier)'
]

hit_ratio_evolucao = {
    'Fases': fases_workload,
    'MRU':   [97.92, 95.97, 91.46, 92.11],  
    'FIFO':  [97.46, 98.73, 98.18, 98],  
    # 'LRU':   [97.46, 98.73, 99.18, 98],  
    'LRU':   [84.75, 84.75, 84.03, 84.21],  
    # 'LRU-K': [97.46, 98.73, 97.2, 97.05]   
    'LRU-K': [91.53, 93.22, 91.32, 91.37]   
}

hit_count_evolucao = {
    'Fases': fases_workload,
    'MRU':   [224, 453, 653, 875],  
    'FIFO':  [230, 466, 701, 931],  
    # 'LRU':   [230, 466, 701, 931],  
    'LRU':   [200, 400, 600, 800],  
    # 'LRU-K': [230, 466, 694, 922]   
    'LRU-K': [216, 440, 652, 868]   
}

miss_count_evolucao = {
    'Fases': fases_workload,
    'MRU':   [12, 19, 61, 75],  
    'FIFO':  [6, 6, 13, 19],  
    # 'LRU':   [6, 6, 13, 19],  
    'LRU':   [36, 72, 114, 150],  
    # 'LRU-K': [6, 6, 20, 28]   
    'LRU-K': [20, 32, 62, 82]   
}

operations_count_evolucao = {
    'Fases': fases_workload,
    'MRU':   [236, 472, 714, 950],  
    'FIFO':  [236, 472, 714, 950],  
    # 'LRU':   [236, 472, 714, 950],  
    'LRU':   [236, 472, 714, 950],  
    # 'LRU-K': [236, 472, 714, 950]   
    'LRU-K': [236, 472, 714, 950]   
}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Estruturação dos Dados: Evolução do Hit Ratio ao longo das fases de execução
# Substitua estes valores pelos que recolher no Seal-DB ao observar a interface passo a passo
fases_workload = [
    '1. Carga Inicial\n(SELECT employee)', 
    '2. Releitura\n(SELECT employee)', 
    '3. Sequential Flooding\n(SELECT sales)', 
    '4. Prova de Retenção\n(SELECT employee)'
]

dados_evolucao = {
    'Fases': fases_workload,
    'FIFO':  [0, 40, 25, 15],  # Cai rapidamente e não recupera
    'LRU':   [0, 45, 18, 12],  # O pior cenário no Sequential Flooding
    'MRU':   [0, 35, 20, 18],  # Errático
    'LRU-K': [0, 45, 42, 65]   # Blindado contra a varredura, recupera com força
}

df_evolucao = pd.DataFrame(dados_evolucao)

# Transformar os dados (Melt) para o formato longo, ideal para o Seaborn
df_melted = df_evolucao.melt(id_vars='Fases', var_name='Algoritmo', value_name='Hit Ratio (%)')

# 2. Configuração do Gráfico de Linhas (Série Temporal do Buffer)
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Traçar as linhas de evolução
ax = sns.lineplot(
    data=df_melted, 
    x='Fases', 
    y='Hit Ratio (%)', 
    hue='Algoritmo', 
    marker='o', 
    linewidth=3, 
    markersize=10,
    palette=['#7f8c8d', '#e74c3c', '#f39c12', '#2ecc71'] # LRU-K em destaque (Verde)
)

# 3. Anotações e Estilização para o Relatório
plt.title('Degradação de Memória: Análise de Sequential Flooding no Seal-DB', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Fases do Workload', fontsize=12, labelpad=15)
plt.ylabel('Taxa de Acerto - Hit Ratio (%)', fontsize=12)
plt.ylim(-5, 100)

# Destacar a área de colapso do LRU
plt.axvspan(1.8, 2.2, color='red', alpha=0.1)
plt.text(2, 80, 'Impacto do\nSequential Flooding', color='red', horizontalalignment='center', fontsize=10, fontweight='bold')

# Adicionar rótulos de dados nos pontos do LRU e LRU-K para contraste
for idx, row in df_evolucao.iterrows():
    plt.text(idx, row['LRU-K'] + 3, f"{row['LRU-K']}%", color='#27ae60', ha='center', fontweight='bold')
    plt.text(idx, row['LRU'] - 5, f"{row['LRU']}%", color='#c0392b', ha='center', fontweight='bold')

plt.legend(title='Políticas de Buffer', fontsize=11, title_fontsize=12, loc='upper left')
plt.tight_layout()
plt.show()